## Line plots
This notebook creates simple line plots to summarize the effect of three different link formation mechanisms on various network inequality metrics.
The link formation mechanisms are
- random `U`: Choose target node at random
- homophily `H`: Choose target node based on its group membership
- preferential attachment and homophily `PAH`: Choose target node based on its popularity
In addition, nodes may be limited by triadic closure, selecting only locally among their friends of friends or globally among all available nodes.

For each metric, we varying the link formation mechanisms per triadic closure and global scope, considering five combinations (`global, triadic closure`):
1. (R,R)
2. (H,R)
3. (H,H)
4. (PAH, R)
5. (PAH, PAH) 

The metrics are
- `gini`: Global degree inequality
- `ei`: Network segregation as ratio of out- and in-group links
- `mann_whitney`: Measures whether degrees of minority nodes tend to be higher than majority node degrees

As an input it takes aggregated statistics from a `.csv`-file.


### Imports and configuration

In [ ]:
from typing import Optional, Tuple, List, Dict, Any
from itertools import product
import os
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import pandas as pd
from netin.models import CompoundLFM
import numpy as np

from patch.constants import\
    PATH_STATISTICS, PATH_PLOTS, PATH_GRAPHS,\
    N, F, L_HOMOPHILY, L_TAU,\
    L_LFM_LOCAL, L_LFM_GLOBAL, N_REALIZATIONS,\
    SIZE_FIG, MAP_LFM_SHORT, MAP_STAT_LABEL,\
    MAP_CM_H
from patch.io import gen_nets_from_file
from patch.statistics import get_cdf

Configuration and utility data structures to make the plotting easier.

In [ ]:
MAP_LFM_AX = {
    (CompoundLFM.HOMOPHILY.value, CompoundLFM.UNIFORM.value) : (0, 0),
    (CompoundLFM.HOMOPHILY.value, CompoundLFM.HOMOPHILY.value) : (1, 0),
    (CompoundLFM.PAH.value, CompoundLFM.UNIFORM.value) : (0, 1),
    (CompoundLFM.PAH.value, CompoundLFM.PAH.value) : (1, 1),
}

plt.rcParams["font.size"] = 10
plt.rcParams["figure.figsize"] = (SIZE_FIG[0], (2/3)*SIZE_FIG[1])
plt.rcParams["axes.labelpad"] = .5
plt.rcParams["axes.titlesize"] = 10
plt.rcParams['figure.constrained_layout.use'] = True

In [ ]:
M = 3

In [ ]:
path_graphs = os.path.join("..", PATH_GRAPHS, f"N-{N}_m-{M}_f-{F}")
print(f"Loading graphs from {path_graphs}")

### Data
Read the data from the provided `.csv`-file.

In [ ]:
data = pd\
    .read_csv(os.path.join("..", PATH_STATISTICS, f"N-{N}_m-{M}_f-{F}.csv"))\
    .set_index(["lfm_global", "lfm_tc", "homophily", "tau", "realization"])\
    .sort_index()
data

Derive some constants (values for homophily, triadic closure and number of realizations) from the data table.

In [ ]:
TAU_VALS = data.index.get_level_values("tau").unique().sort_values()
H_VALS = data.index.get_level_values("homophily").unique().sort_values()
N_REAL = data.index.get_level_values("realization").nunique()

print(f"homophily values: {H_VALS.values}")
assert all(h_val in L_HOMOPHILY for h_val in H_VALS) and len(L_HOMOPHILY) == len(H_VALS)

print(f"triadic closure values: {TAU_VALS.values}")
assert all(tau_val in L_TAU for tau_val in TAU_VALS) and len(L_TAU) == len(TAU_VALS)

print(f"number of realizations: {N_REAL}")
assert N_REAL == N_REALIZATIONS

In [ ]:
TAU_VALS_FILTERED = [TAU_VALS[0], TAU_VALS[3]]
print(f"filtered triadic closure values: {TAU_VALS_FILTERED}")

In [ ]:
data["gini_ratio"] = data["gini_min"] / data["gini_maj"]

### Plotting
Create, for each metric, a grid of five plots based on the combinations of local and global link formation mechanisms.

In [ ]:
def _plot_custom_legend(
        fig: plt.figure
):
    # Add horizontal legend at the top of the plot showing the colors of varying homophily values and the markers for the different tau values
    l_artists = []
    for h in H_VALS:
        l_artists.append(plt.Line2D([0], [0], color=MAP_CM_H[h], label=f"$h={h}$"))
    l_artists.extend([
        plt.Line2D([0], [0], color="black", marker=marker, linestyle="none", label=f"$\\tau={tau}$")\
            for marker, tau in zip(("x", "^"), TAU_VALS_FILTERED)])
    fig.legend(
        handles=l_artists,
        loc="upper center",
        # Reduce length of handle lines
        handlelength=1.,
        ncol=len(H_VALS) + 2,
        frameon=False,
        columnspacing=0.5,
        handletextpad=0.2,
        bbox_to_anchor=(0.5, 1.1))

In [ ]:
def _plot_single_lfm_comb(
        lfm_global: str, lfm_tc: str,
        data_metric: pd.DataFrame,
        ax: plt.Axes,
        grouping_key: str = "homophily",
        grouping_vals: Optional[List[float]] = H_VALS,
        plot_label: bool = False,
        plot_title: bool = False,
        vneutral: Optional[float] = None,
        pos_lfm_label: Tuple[float, float] = (0.05, 0.9),
        l_kwargs: Optional[List[Dict[str, Any]]] = None):
    for group, kwargs in zip(grouping_vals, l_kwargs):
        data_aggregates = data_metric\
            .xs((lfm_global, lfm_tc, group),
                level=("lfm_global", "lfm_tc", grouping_key))\
            .groupby(
                axis="index",
                level="tau" if grouping_key == "homophily" else "homophily")\
            .aggregate(["mean", "std"]) # Compute mean and std

        # Plot mean with std error
        ax.errorbar(
            x=TAU_VALS,
            y=data_aggregates["mean"],
            yerr=data_aggregates["std"],
            label=(f"$h={group}$"\
                   if grouping_key == "homophily" else f"$\\tau={group}$")
                    if plot_label else None,
            **kwargs)

    if vneutral is not None:
        # Plot neutral metric line if provided
        ax.axhline(vneutral, color="black", linestyle="--")

    if plot_title:
        ax.text(
            *pos_lfm_label,
            f"${MAP_LFM_SHORT[lfm_global]}$, ${MAP_LFM_SHORT[lfm_tc]}$",
            horizontalalignment="left" if pos_lfm_label[0] < 0.5 else "right",
            transform=ax.transAxes)


In [ ]:
def plot_metric_over_h_sub(
        ax: plt.Axes,
        lfm_global: str, lfm_tc: str,
        data: pd.DataFrame,
        metric: str):
    for h in H_VALS:
        ax.axvline(h, color=MAP_CM_H[h], linestyle=":")
    _plot_single_lfm_comb(
        lfm_global=lfm_global, lfm_tc=lfm_tc,
        data_metric=data[metric],
        ax=ax,
        plot_title=False,
        plot_label= (lfm_global == L_LFM_GLOBAL[0] and lfm_tc == L_LFM_LOCAL[0]),
        grouping_key="tau",
        grouping_vals=TAU_VALS_FILTERED,
        l_kwargs=[{
            "color": "black",
            "marker": marker,
            "linestyle": "none",
        } for marker in ("x", "^")])
    ax.set_xticks(H_VALS[::2])
    ax.spines[["top", "right"]].set_visible(False)

In [ ]:
def plot_metric_over_tau(
    metric: str,
    data: pd.DataFrame,
    vneutral: Optional[float] = None,
    pos_lfm_label: Tuple[float, float] = (0.05, 0.9))\
        -> Tuple[plt.Figure, np.ndarray]:
    assert metric in data.columns,\
        f"Could not find metric `{metric}` in data columns `{data.columns}`"

    # Create plot
    fig = plt.figure()
    gs = gridspec.GridSpec(
        nrows=2, ncols=5, figure=fig,
        width_ratios=[2,1,.15,2,1],
        height_ratios=[1,1])

    _ax_00 = fig.add_subplot(gs[0,0])
    a_ax_tau = np.array([
        [_ax_00, fig.add_subplot(gs[0,-2], sharey=_ax_00)],
        [
            fig.add_subplot(gs[1,0], sharey=_ax_00),
            fig.add_subplot(gs[1,-2], sharey=_ax_00)]])
    a_ax_h = np.array([
        [
            fig.add_subplot(gs[0,1], sharey=_ax_00),
            fig.add_subplot(gs[0,-1], sharey=_ax_00)],
        [
            fig.add_subplot(gs[1,1], sharey=_ax_00),
            fig.add_subplot(gs[1,-1], sharey=_ax_00)]])


    for lfm_global, lfm_tc in product(L_LFM_GLOBAL, L_LFM_LOCAL):
        if ((lfm_global, lfm_tc) not in data.index):
            continue # Skip invalid combinations or U,U
        ax = a_ax_tau[MAP_LFM_AX[(lfm_global, lfm_tc)]]
        _plot_single_lfm_comb(
            lfm_global=lfm_global, lfm_tc=lfm_tc,
            data_metric=data[metric],
            ax=ax,
            plot_title=True,
            plot_label=(lfm_global, lfm_tc) == (CompoundLFM.HOMOPHILY.value, CompoundLFM.UNIFORM.value),
            vneutral=vneutral,
            pos_lfm_label=pos_lfm_label,
            grouping_key="homophily",
            grouping_vals=H_VALS,
            l_kwargs=[{
                "color": MAP_CM_H[h],
                "marker": "s"
            } for h in H_VALS])

        ax_h = a_ax_h[MAP_LFM_AX[(lfm_global, lfm_tc)]]
        plot_metric_over_h_sub(
            ax=ax_h,
            lfm_global=lfm_global,
            lfm_tc=lfm_tc,
            data=data,
            metric=metric)

    for ax in a_ax_h.flatten():
        ax.set_xlim(-.1, ax.get_xlim()[1])
        plt.setp(ax.get_yticklabels(), visible=False)
    for ax in a_ax_tau[:,1]:
        plt.setp(ax.get_yticklabels(), visible=False)

    for ax in a_ax_tau[:,0]:
        ax.set_ylabel(MAP_STAT_LABEL[metric])

    for ax in a_ax_h[-1]:
        ax.set_xlabel(r"$h$")

    for ax in a_ax_tau[-1]:
        ax.set_xlabel(r"$\tau$")

    for a_ax in (a_ax_tau, a_ax_h):
        for ax in a_ax[0]:
            ax.set_xticks([])
        for ax in a_ax.flatten():
            ax.spines[["top", "right"]].set_visible(False)

    for label, pos in zip(("a", "b", "c", "d"), ((0.0, 0.96), (0.5125, 0.96), (0.0, 0.54), (0.5125, 0.54))):
        fig.text(*pos, label, fontweight="bold")

    # Add horizontal legend at the top of the plot
    fig.legend(
        loc="upper center",
        ncol=len(H_VALS) + 2,
        frameon=False,
        columnspacing=0.5,
        handletextpad=0.2,
        bbox_to_anchor=(0.5, 1.1))

    # fig.tight_layout()

    return fig, a_ax

In [ ]:
def plot_gini_scatter(
    data: pd.DataFrame,
    pos_lfm_label: Tuple[float, float] = (0.0, 0.05))\
        -> Tuple[plt.Figure, np.ndarray]:
    # Create plot
    fig = plt.figure()
    gs = gridspec.GridSpec(
        nrows=2, ncols=4, figure=fig,
        width_ratios=[1,1,2,2],
        height_ratios=[1,1])

    _ax_00 = fig.add_subplot(gs[0,0])
    a_ax_h = np.array([
        [_ax_00, fig.add_subplot(gs[0,1], sharex=_ax_00, sharey=_ax_00)],
        [
            fig.add_subplot(gs[1,0], sharex=_ax_00, sharey=_ax_00),
            fig.add_subplot(gs[1,1], sharex=_ax_00, sharey=_ax_00)]])
    _ax_12 = fig.add_subplot(gs[1,2], sharey=_ax_00)
    _ax_13 = fig.add_subplot(gs[1,3], sharex=_ax_12, sharey=_ax_00)
    a_ax_gini = np.array([
        [
            fig.add_subplot(gs[0,2], sharex=_ax_12, sharey=_ax_00),
            fig.add_subplot(gs[0,3], sharex=_ax_12, sharey=_ax_00)],
        [
            _ax_12,
            _ax_13]])

    for lfm_global, lfm_tc in product(L_LFM_GLOBAL, L_LFM_LOCAL):
        if ((lfm_global, lfm_tc) not in data.index):
            continue # Skip invalid combinations or U,U
        ax_gini = a_ax_gini[MAP_LFM_AX[(lfm_global, lfm_tc)]]
        # Plot two single error bar markers for each homophily value
        # The x- and y-position of each marker is determined by the mean `gini_minority` and `gini_majority` values
        # Errorbars should be standard deviations across realizations
        # One marker is for tau=0.0, the other for tau=0.75
        for h in H_VALS:
            data_aggregates = data\
                .xs((lfm_global, lfm_tc, h),
                    level=("lfm_global", "lfm_tc", "homophily"))\
                .groupby(axis="index", level="tau")\
                .aggregate(["mean", "std"])
            ax_gini.errorbar(
                x=data_aggregates.loc[0.0, "gini_min"]["mean"],
                y=data_aggregates.loc[0.0, "gini_maj"]["mean"],
                xerr=data_aggregates.loc[0.0, "gini_min"]["std"],
                yerr=data_aggregates.loc[0.0, "gini_maj"]["std"],
                color=MAP_CM_H[h],
                marker="x",
                linestyle="none")
            ax_gini.errorbar(
                x=data_aggregates.loc[0.75, "gini_min"]["mean"],
                y=data_aggregates.loc[0.75, "gini_maj"]["mean"],
                xerr=data_aggregates.loc[0.75, "gini_min"]["std"],
                yerr=data_aggregates.loc[0.75, "gini_maj"]["std"],
                color=MAP_CM_H[h],
                marker="^",
                linestyle="none")
        # Plot diagonal line at min(gini, gini_m, gini_M)
        _min = min(data["gini_min"].min(), data["gini_maj"].min(), data["gini"].min())
        ax_gini.axline([_min, _min], slope=1, color="black", linestyle="--")

        ax_gini.text(
            pos_lfm_label[0],
            pos_lfm_label[1],
            f"${MAP_LFM_SHORT[lfm_global]}$, ${MAP_LFM_SHORT[lfm_tc]}$",
            transform=ax_gini.transAxes)

        ax_h = a_ax_h[MAP_LFM_AX[(lfm_global, lfm_tc)]]
        plot_metric_over_h_sub(
            ax=ax_h,
            lfm_global=lfm_global,
            lfm_tc=lfm_tc,
            data=data,
            metric="gini")
        ax_h.text(
            pos_lfm_label[0] + .05,
            pos_lfm_label[1],
            f"${MAP_LFM_SHORT[lfm_global]}$, ${MAP_LFM_SHORT[lfm_tc]}$",
            # White background
            bbox=dict(
                facecolor="white",
                alpha=0.75,
                edgecolor="white",
                # Reduce size
                pad=.1
                ),
            transform=ax_h.transAxes)

    for ax in a_ax_gini.flatten():
        # ax.set_xlim(-.15, 1.15)
        # Set y-axis to right hand side
        # Rotate y-axis label and set position
        ax.yaxis.set_label_position("right")
        ax.spines[["top", "left"]].set_visible(False)
        ax.yaxis.tick_right()

    for ax in a_ax_gini[:,0]:
        plt.setp(ax.get_yticklabels(), visible=False)

    for ax in a_ax_gini[:,-1]:
        ax.set_ylabel(r"$\mathregular{Gini_M}$", labelpad=12)
        ax.yaxis.label.set_rotation(270)

    for ax in a_ax_gini[-1]:
        ax.set_xlabel(r"$\mathregular{Gini_m}$")

    for ax in a_ax_h[:,0]:
        ax.set_ylabel(MAP_STAT_LABEL["gini"], labelpad=2)

    for ax in a_ax_h[:,1]:
        plt.setp(ax.get_yticklabels(), visible=False)

    for ax in a_ax_h[-1]:
        ax.set_xlabel(r"$h$")

    for ax in a_ax_h.flatten():
        ax.set_xlim(-.125, 1.125)
        ax.spines[["top", "right"]].set_visible(False)

    for a_ax in (a_ax_gini, a_ax_h):
        for ax in a_ax[0]:
            plt.setp(ax.get_xticklabels(), visible=False)

    _ax_12.set_xlim(0.2, _ax_12.get_xlim()[1])

    for label, pos in zip(("a", "b"), ((0.0, 0.925), (0.375, 0.925))):
        fig.text(*pos, label, fontweight="bold")

    _plot_custom_legend(fig)

    return fig, a_ax

In [ ]:
def plot_gini_mw_scatter(
    data: pd.DataFrame)\
        -> Tuple[plt.Figure, np.ndarray]:
    fig = plt.figure()
    gs = gridspec.GridSpec(
        nrows=2, ncols=4, figure=fig,
        width_ratios=[1,1,1,1],
        height_ratios=[1,1])

    _ax_00 = fig.add_subplot(gs[0,0])
    a_ax_gini = np.array([
        [_ax_00, fig.add_subplot(gs[0,1], sharex=_ax_00, sharey=_ax_00)],
        [
            fig.add_subplot(gs[1,0], sharex=_ax_00, sharey=_ax_00),
            fig.add_subplot(gs[1,1], sharex=_ax_00, sharey=_ax_00)]])

    _ax_01 = fig.add_subplot(gs[0,2], sharex=_ax_00)
    a_ax_gini_ratio = np.array([
        [
            _ax_01,
            fig.add_subplot(gs[0,3], sharex=_ax_00, sharey=_ax_01)],
        [
            fig.add_subplot(gs[1,2], sharex=_ax_00, sharey=_ax_01),
            fig.add_subplot(gs[1,3], sharex=_ax_00, sharey=_ax_01),]])

    for lfm_global, lfm_tc in product(L_LFM_GLOBAL, L_LFM_LOCAL):
        if ((lfm_global, lfm_tc) not in data.index):
            continue
        for a_ax, metric_gini in zip((a_ax_gini, a_ax_gini_ratio), ("gini", "gini_ratio")):
            ax = a_ax[MAP_LFM_AX[(lfm_global, lfm_tc)]]
            for h in H_VALS:
                data_aggregates = data\
                    .xs((lfm_global, lfm_tc, h),
                        level=("lfm_global", "lfm_tc", "homophily"))\
                    .groupby(axis="index", level="tau")\
                    .aggregate(["mean", "std"])
                ax.errorbar(
                    x=data_aggregates.loc[0.0, "mann_whitney"]["mean"],
                    y=data_aggregates.loc[0.0, metric_gini]["mean"],
                    xerr=data_aggregates.loc[0.0, "mann_whitney"]["std"],
                    yerr=data_aggregates.loc[0.0, metric_gini]["std"],
                    color=MAP_CM_H[h],
                    marker="x",
                    linestyle="none")
                ax.errorbar(
                    x=data_aggregates.loc[0.75, "mann_whitney"]["mean"],
                    y=data_aggregates.loc[0.75, metric_gini]["mean"],
                    xerr=data_aggregates.loc[0.75, "mann_whitney"]["std"],
                    yerr=data_aggregates.loc[0.75, metric_gini]["std"],
                    color=MAP_CM_H[h],
                    marker="^",
                    linestyle="none")

    # Iterate over all axes
    for a_ax, metric_gini in zip((a_ax_gini, a_ax_gini_ratio), ("gini", "gini_ratio")):
        for ax in a_ax.flatten():
            ax.axvline(.5, color="black", linestyle="--")
        for ax in a_ax[-1]:
            ax.set_xlabel("Mann-Whitney")
        for ax in a_ax[0]:
            plt.setp(ax.get_xticklabels(), visible=False)

    # Iterate over gini axes
    for ax in a_ax_gini[:,0]:
        ax.set_ylabel(MAP_STAT_LABEL["gini"])
    for ax in a_ax_gini[:,-1]:
        plt.setp(ax.get_yticklabels(), visible=False)
    for ax in a_ax_gini.flatten():
        ax.spines[["top", "right"]].set_visible(False)

    # Iterate over gini-ratio axes
    for ax in a_ax_gini_ratio.flatten():
        ax.axhline(1, color="black", linestyle="--")
        ax.set_yscale("symlog", linthresh=.01)
        ax.spines[["top", "left"]].set_visible(False)
        ax.yaxis.tick_right()

    for ax in a_ax_gini_ratio[:,1]:
        ax.set_yticks([0.1, 1, 10])
        ax.set_yticks(np.logspace(np.log10(.1), np.log10(10), 20), minor=True)
        ax.set_ylabel(
            "$\\mathregular{Gini_{m}} / \\mathregular{Gini_{M}}$", labelpad=12)
        # Move y axis right
        ax.yaxis.set_label_position("right")
        ax.yaxis.label.set_rotation(270)

    for ax in a_ax_gini_ratio[:,0]:
        # Remove labels
        plt.setp(ax.get_yticklabels(), visible=False)


    # Iterate over LFM models
    for lfm_global, lfm_tc in product(L_LFM_GLOBAL, L_LFM_LOCAL):
        if ((lfm_global, lfm_tc) not in data.index):
            continue
        for a_ax in (a_ax_gini, a_ax_gini_ratio):
            ax = a_ax[MAP_LFM_AX[(lfm_global, lfm_tc)]]
            ax.text(
                .975, .05, # bottom right corner
                f"${MAP_LFM_SHORT[lfm_global]}$, ${MAP_LFM_SHORT[lfm_tc]}$",
                horizontalalignment="right",
                transform=ax.transAxes)

    for label, pos in zip(("a", "b"), ((0.0, 0.925), (0.475, 0.925))):
        fig.text(*pos, label, fontweight="bold")

    _plot_custom_legend(fig)

    return fig, a_ax


#### Plots per metric
Create plots for each metric.

In [ ]:
# Filter data by metric column
metric = "gini"

fig, a_ax = plot_metric_over_tau(metric=metric, data=data)

file = os.path.join("../", PATH_PLOTS, f"{metric}_N-{N}_f-{F}_m-{M}.pdf")
fig.savefig(file, bbox_inches="tight")
print(f"Saving file to `{file}`.")

In [ ]:
fig, a_ax = plot_gini_scatter(data=data)
file = os.path.join("../", PATH_PLOTS, f"gini_scatter_N-{N}_f-{F}_m-{M}.pdf")
fig.savefig(file, bbox_inches="tight")
print(f"Saving file to `{file}`.")

In [ ]:
fig, a_ax = plot_gini_mw_scatter(data=data)
file = os.path.join("../", PATH_PLOTS, f"gini_mw_scatter_N-{N}_f-{F}_m-{M}.pdf")
fig.savefig(file, bbox_inches="tight")
print(f"Saving file to `{file}`.")

In [ ]:
# Filter data by metric column
metric = "ei"

fig, a_ax = plot_metric_over_tau(
    metric=metric,
    data=data,
    vneutral=0.,
    pos_lfm_label=(0.975, 0.9))

file = os.path.join("../", PATH_PLOTS, f"{metric}_N-{N}_f-{F}_m-{M}.pdf")
fig.savefig(file, bbox_inches="tight")
print(f"Saving file to `{file}`.")

In [ ]:
# Filter data by metric column
metric = "mann_whitney"

fig, a_ax = plot_metric_over_tau(metric=metric, data=data, vneutral=0.5, pos_lfm_label=(0.975, 0.9))

file = os.path.join("../", PATH_PLOTS, f"{metric}_N-{N}_f-{F}_m-{M}.pdf")
fig.savefig(file, bbox_inches="tight")
print(f"Saving file to `{file}`.")